In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import Docx2txtLoader

In [2]:
def load_pdf_files(data):
    Loaders =[
        DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader),
        DirectoryLoader(data, glob="*.docx", loader_cls=Docx2txtLoader),
        DirectoryLoader(data, glob="*.txt", loader_cls=TextLoader)
    ]
    documents =[]
    for loader in Loaders:
        documents.extend(loader.load())
    return documents

In [3]:
extracted_data = load_pdf_files("data")

In [4]:
extracted_data

[Document(metadata={'source': 'data\\Reinforcement Learning Assignment Report.docx'}, page_content="Reinforcement Learning Assignment Report\n\nIntroduction\n\nThis report presents the results of a reinforcement learning assignment that includes two main parts: Q-Learning and Policy Iteration on the FrozenLake environment, and Deep Q-Learning on Atari Breakout. The experiments show how different RL algorithms perform and provide useful insights into how sensitive they are to hyperparameters and how they compare with each other.\n\nPart 1: Q-Learning and Policy Iteration on Frozen Lake Environment\n\nEnvironment Overview\n\nThe Frozen Lake environment is a classic grid-world problem with the following characteristics:\n\nState space: 16 discrete states (4×4 grid)\n\nAction space: 4 actions (Up, Down, Left, Right)\n\nEnvironment type: Stochastic (slippery surface)\n\nObjective: Navigate from start to goal while avoiding holes\n\n\n\nAlgorithm Implementation\n\nQ-Learning Algorithm\n\nThe

In [5]:
len(extracted_data)

1

In [6]:
from langchain.schema import Document
import re

def clean_text(text):
    # Remove hyphenation at line breaks
    text = re.sub(r'-\n', '', text)
    # Replace line breaks with space
    text = re.sub(r'\n+', ' ', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Optionally remove page numbers (assuming isolated numbers)
    text = re.sub(r'\s\d+\s', ' ', text)
    return text.strip()

def clean_data(documents):
    cleaned_docs = []
    for doc in documents:
        cleaned_content = clean_text(doc.page_content)
        if len(cleaned_content) > 100:  # filter short/empty pages
            cleaned_docs.append(
                Document(
                    page_content=cleaned_content,
                    metadata={
                        'source': doc.metadata.get('source'),
                        'page': doc.metadata.get('page')
                    }
                )
            )
    return cleaned_docs


In [7]:
cleaned_data = clean_data(extracted_data)

In [8]:
cleaned_data

[Document(metadata={'source': 'data\\Reinforcement Learning Assignment Report.docx', 'page': None}, page_content="Reinforcement Learning Assignment Report Introduction This report presents the results of a reinforcement learning assignment that includes two main parts: Q-Learning and Policy Iteration on the FrozenLake environment, and Deep Q-Learning on Atari Breakout. The experiments show how different RL algorithms perform and provide useful insights into how sensitive they are to hyperparameters and how they compare with each other. Part 1: Q-Learning and Policy Iteration on Frozen Lake Environment Environment Overview The Frozen Lake environment is a classic grid-world problem with the following characteristics: State space: discrete states (4×4 grid) Action space: actions (Up, Down, Left, Right) Environment type: Stochastic (slippery surface) Objective: Navigate from start to goal while avoiding holes Algorithm Implementation Q-Learning Algorithm The Q-Learning agent was implement

In [ ]:
def text_split(cleaned_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200,
        length_function = len
    )
    texts = text_splitter.split_documents(cleaned_data)
    return texts

In [10]:
texts = text_split(cleaned_data)

In [11]:
len(texts)

18

In [12]:
texts

[Document(metadata={'source': 'data\\Reinforcement Learning Assignment Report.docx', 'page': None}, page_content='Reinforcement Learning Assignment Report Introduction This report presents the results of a reinforcement learning assignment that includes two main parts: Q-Learning and Policy Iteration on the FrozenLake environment, and Deep Q-Learning on Atari Breakout. The experiments show how different RL algorithms perform and provide useful insights into how sensitive they are to hyperparameters and how they compare with each other. Part 1: Q-Learning and Policy Iteration on Frozen Lake Environment'),
 Document(metadata={'source': 'data\\Reinforcement Learning Assignment Report.docx', 'page': None}, page_content='Lake Environment Environment Overview The Frozen Lake environment is a classic grid-world problem with the following characteristics: State space: discrete states (4×4 grid) Action space: actions (Up, Down, Left, Right) Environment type: Stochastic (slippery surface) Object

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings


def download_embedding():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embedding = HuggingFaceEmbeddings(
        model_name = model_name
    )
    return embedding

In [14]:
embedding = download_embedding()

C:\Users\nanda\AppData\Local\Temp\ipykernel_19052\1886361177.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


In [15]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [16]:
vector = embedding.embed_query("Hello World")

In [17]:
len(vector)

384

In [18]:
from langchain_community.vectorstores import Chroma

class VectorStore:
    def __init__(self, path):
        self.embedding = download_embedding()
        self.vector_store = Chroma(
            persist_directory = path,
            embedding_function = self.embedding
        )

    def add_documents(self, documents):
        self.vector_store.add_documents(documents)

 

In [19]:
VECTOR_DB_PATH = 'vector_db'

In [20]:
vector_store = VectorStore(VECTOR_DB_PATH)
vector_store.add_documents(texts)

C:\Users\nanda\AppData\Local\Temp\ipykernel_19052\1257519252.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  self.vector_store = Chroma(


In [21]:
retriever=vector_store.vector_store.as_retriever(
                    search_type="similarity",
                    search_kwargs={"k": 4} 
                )

In [26]:
retrieved_docs=retriever.invoke("what is the average reward of policy iteration ?")

In [27]:
retrieved_docs

[Document(metadata={'source': 'data\\Reinforcement Learning Assignment Report.docx'}, page_content='values were tested: 0.1, 0.5, and 1.0 Results: Epsilon = 0.1: Average Reward = 0.000| Success Rate = 0.0% Epsilon = 0.5: Average Reward = 0.000| Success Rate = 0.0% Epsilon = 1.0: Average Reward = 0.720| Success Rate = 72.0% Analysis: High initial exploration (ε=1.0) was crucial for learning an effective policy. Lower exploration rates failed completely, highlighting the importance of exploration in stochastic environments. Experimental Results Algorithm Comparison The comparison between'),
 Document(metadata={'source': 'data\\Reinforcement Learning Assignment Report.docx'}, page_content='comparison between Q-Learning and Policy Iteration revealed significant performance differences: The performance of Q-Learning and Policy Iteration was compared in the FrozenLake environment. Q-Learning achieved an average reward of 0.610 with a success rate of 61.0%. In contrast, Policy Iteration perfo